# CADD v1.7 Benchmark — TP53 Held-Out Test SetReproduces the CADD v1.7 comparison reported in Table 4 of the manuscript (AUC-ROC = 0.9814, n = 294). Requires `clinvar_result.txt` (raw ClinVar export) in the working directory.**Step 1** rebuilds the exact dataset and test split used to train the RF model. **Step 2** exports a VCF for CADD submission. **Step 3** (manual) submits the VCF to the CADD scoring server. **Step 4** merges the scored results and computes AUC.

## Step 1: Rebuild dataset and reproduce the exact test split

In [ ]:
import pandas as pdfrom sklearn.model_selection import train_test_splitdf_raw = pd.read_csv('clinvar_result.txt', sep='\t')print(f"df_raw shape: {df_raw.shape}")keep_labels = [    'Pathogenic', 'Likely pathogenic', 'Pathogenic/Likely pathogenic',    'Pathogenic/Likely pathogenic/Pathogenic, low penetrance',    'Benign', 'Likely benign', 'Likely benign/Benign', 'Benign/Likely benign']df_raw_filtered = df_raw[df_raw['Germline classification'].isin(keep_labels)].reset_index(drop=True)pathogenic_set = {'Pathogenic', 'Likely pathogenic', 'Pathogenic/Likely pathogenic',                   'Pathogenic/Likely pathogenic/Pathogenic, low penetrance'}df = df_raw_filtered.copy()df['label'] = df['Germline classification'].apply(lambda x: 1 if x in pathogenic_set else 0)assert len(df) == 1470, "Row count mismatch — stop and recheck the ClinVar export"print(f"df rows: {len(df)} | Pathogenic: {(df['label']==1).sum()} | Benign: {(df['label']==0).sum()}")

In [ ]:
_, X_test_df, _, _ = train_test_split(    df, df['label'], test_size=0.2, random_state=42, stratify=df['label'])test_indices = X_test_df.indexprint(f"Test set size: {len(test_indices)}")test_labels = pd.DataFrame({    'Pos': df_raw_filtered.loc[test_indices, 'Canonical SPDI'].apply(lambda x: int(x.split(':')[1]) + 1),    'Ref': df_raw_filtered.loc[test_indices, 'Canonical SPDI'].apply(lambda x: x.split(':')[2]),    'Alt': df_raw_filtered.loc[test_indices, 'Canonical SPDI'].apply(lambda x: x.split(':')[3]),    'label': df.loc[test_indices, 'label'].values})print(f"Pathogenic: {(test_labels['label']==1).sum()} | Benign: {(test_labels['label']==0).sum()}")test_labels.to_csv('test_labels_294.csv', index=False)print("Saved: test_labels_294.csv")

## Step 2: Export VCF for CADD submission

In [ ]:
vcf_rows = [f"17\t{row['Pos']}\t.\t{row['Ref']}\t{row['Alt']}" for _, row in test_labels.iterrows()]with open('tp53_testset_294.vcf', 'w') as f:    f.write('\n'.join(vcf_rows) + '\n')print(f"Saved: tp53_testset_294.vcf ({len(vcf_rows)} variants)")

## Step 3: Submit to CADD (manual)Upload `tp53_testset_294.vcf` to the [CADD scoring server](https://cadd.gs.washington.edu/score), selecting **GRCh38-v1.7**. Download the scored TSV when complete and place it in the working directory.

## Step 4: Merge CADD scores against test labels and compute AUC

In [ ]:
from sklearn.metrics import roc_auc_score, confusion_matrix# Update this filename to match your downloaded CADD outputcadd_path = 'GRCh38-v1.7_scored.tsv'cadd = pd.read_csv(cadd_path, sep='\t', comment='#', header=None,                    names=['Chrom', 'Pos', 'Ref', 'Alt', 'RawScore', 'PHRED'])cadd['Pos'] = pd.to_numeric(cadd['Pos'], errors='coerce')cadd['PHRED'] = pd.to_numeric(cadd['PHRED'], errors='coerce')print(f"CADD rows: {len(cadd)}")print(f"Duplicate Pos/Ref/Alt in CADD: {cadd.duplicated(subset=['Pos','Ref','Alt']).sum()}")print(f"Duplicate Pos/Ref/Alt in labels: {test_labels.duplicated(subset=['Pos','Ref','Alt']).sum()}")

In [ ]:
merged = test_labels.merge(cadd, on=['Pos', 'Ref', 'Alt'], how='inner')print(f"Merged rows: {len(merged)}")print(f"Pathogenic: {(merged['label']==1).sum()} | Benign: {(merged['label']==0).sum()}")print("\nMean PHRED by label (sanity check — pathogenic should score higher):")print(merged.groupby('label')['PHRED'].mean())y_true = merged['label'].valuesy_score = merged['PHRED'].valuesauc = roc_auc_score(y_true, y_score)print(f"\nAUC-ROC = {auc:.4f}")for threshold in [20, 25, 30]:    y_pred = (y_score >= threshold).astype(int)    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()    sens, spec = tp/(tp+fn), tn/(tn+fp)    print(f"PHRED>={threshold}: Sensitivity={sens:.4f}, Specificity={spec:.4f}")

**Expected output:** n=294, AUC-ROC=0.9814, with sensitivity/specificity of 0.9512/0.9623 (PHRED≥20), 0.7195/0.9953 (PHRED≥25), and 0.3049/1.0000 (PHRED≥30), matching Table 4 of the manuscript.